# Food Price Inflation Tracker - Data Loading & Exploration

## Objective
This notebook loads all converted datasets and performs initial exploratory analysis to understand:
- Data structure and quality
- Time ranges and coverage
- Key commodities and markets
- Trends and patterns

## Datasets
1. **WFP Food Prices (Kenya)** - Comprehensive food price data 2006-2021
2. **Housing Data (Converted)** - Existing housing datasets (for comparison)
3. **Sample Kenya Food Prices** - Generated sample data with trends

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load WFP Kenya Food Prices Dataset

In [ ]:
# Load WFP Kenya food prices
wfp_kenya = pd.read_csv('../data/raw/wfp_food_prices_kenya_full.csv')

print("="*80)
print("WFP KENYA FOOD PRICES DATASET")
print("="*80)
print(f"\nShape: {wfp_kenya.shape}")
print(f"Date Range: {wfp_kenya['mp_year'].min()}-{wfp_kenya['mp_month'].min():02.0f} to {wfp_kenya['mp_year'].max()}-{wfp_kenya['mp_month'].max():02.0f}")
print(f"Commodities: {wfp_kenya['cm_name'].nunique()}")
print(f"Markets: {wfp_kenya['mkt_name'].nunique()}")
print(f"Regions: {wfp_kenya['adm1_name'].nunique()}")
print(f"\nColumns: {list(wfp_kenya.columns)}")
print(f"\nFirst few rows:")
wfp_kenya.head()

In [ ]:
# Data types and missing values
print("DATA QUALITY CHECK")
print("="*80)
print("\nData Types:")
print(wfp_kenya.dtypes)
print("\nMissing Values:")
missing = wfp_kenya.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(missing)
else:
    print("No missing values!")

print("\nBasic Statistics:")
wfp_kenya.describe()

## 2. Top Commodities Analysis

In [ ]:
# Commodity distribution
print("TOP 15 COMMODITIES BY DATA POINTS")
print("="*80)
commodity_counts = wfp_kenya['cm_name'].value_counts().head(15)
print(commodity_counts)

# Visualize
plt.figure(figsize=(12, 8))
commodity_counts.plot(kind='barh', color='steelblue')
plt.title('Top 15 Commodities by Number of Price Records', fontsize=14, fontweight='bold')
plt.xlabel('Number of Records', fontsize=12)
plt.ylabel('Commodity', fontsize=12)
plt.tight_layout()
plt.show()

# All unique commodities
print(f"\n\nALL {wfp_kenya['cm_name'].nunique()} COMMODITIES:")
print("="*80)
for i, commodity in enumerate(sorted(wfp_kenya['cm_name'].unique()), 1):
    print(f"{i}. {commodity}")

## 3. Markets and Regional Analysis

In [ ]:
# Market distribution
print("TOP 15 MARKETS BY DATA POINTS")
print("="*80)
market_counts = wfp_kenya['mkt_name'].value_counts().head(15)
print(market_counts)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Markets
market_counts.plot(kind='barh', ax=ax1, color='coral')
ax1.set_title('Top 15 Markets by Number of Price Records', fontsize=14, fontweight='bold')
ax1.set_xlabel('Number of Records', fontsize=12)
ax1.set_ylabel('Market', fontsize=12)

# Regions
region_counts = wfp_kenya['adm1_name'].value_counts()
region_counts.plot(kind='bar', ax=ax2, color='mediumseagreen')
ax2.set_title('Regional Distribution of Price Records', fontsize=14, fontweight='bold')
ax2.set_xlabel('Region (Admin1)', fontsize=12)
ax2.set_ylabel('Number of Records', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nREGIONAL COVERAGE:")
print(region_counts)

## 4. Price Analysis - Key Staple Foods

In [ ]:
# Focus on key staple foods
staple_foods = [
    'Maize (white) - Retail',
    'Maize - Wholesale',
    'Beans (dry) - Retail',
    'Rice - Retail',
    'Oil (vegetable) - Retail',
    'Milk (cow, pasteurized) - Retail'
]

# Filter for staple foods
staples_df = wfp_kenya[wfp_kenya['cm_name'].isin(staple_foods)].copy()

print(f"STAPLE FOODS DATA")
print("="*80)
print(f"Total records for staple foods: {len(staples_df):,}")
print(f"\nBreakdown by commodity:")
print(staples_df['cm_name'].value_counts())

# Price statistics
print(f"\n\nPRICE STATISTICS BY COMMODITY")
print("="*80)
price_stats = staples_df.groupby('cm_name')['mp_price'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
print(price_stats)

In [ ]:
# Create date column for time series analysis
staples_df['date'] = pd.to_datetime(
    staples_df['mp_year'].astype(int).astype(str) + '-' + 
    staples_df['mp_month'].astype(int).astype(str).str.zfill(2) + '-01'
)

# Average prices over time
price_trends = staples_df.groupby(['date', 'cm_name'])['mp_price'].mean().reset_index()

# Plot price trends
plt.figure(figsize=(16, 10))

for i, commodity in enumerate(staple_foods, 1):
    plt.subplot(2, 3, i)
    data = price_trends[price_trends['cm_name'] == commodity]
    if len(data) > 0:
        plt.plot(data['date'], data['mp_price'], linewidth=2, color='darkblue')
        plt.title(commodity, fontsize=11, fontweight='bold')
        plt.xlabel('Year', fontsize=9)
        plt.ylabel('Price (KES)', fontsize=9)
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)

plt.suptitle('Price Trends for Key Staple Foods in Kenya (2006-2021)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n✓ Price trends plotted successfully!")

## 5. Inflation Analysis - Year-over-Year Changes

In [ ]:
# Calculate year-over-year inflation for Maize (most important staple)
maize_retail = wfp_kenya[wfp_kenya['cm_name'] == 'Maize (white) - Retail'].copy()
maize_retail['date'] = pd.to_datetime(
    maize_retail['mp_year'].astype(int).astype(str) + '-' + 
    maize_retail['mp_month'].astype(int).astype(str).str.zfill(2) + '-01'
)

# Monthly average prices
maize_monthly = maize_retail.groupby('date')['mp_price'].mean().reset_index()
maize_monthly = maize_monthly.sort_values('date')

# Calculate YoY inflation
maize_monthly['price_12m_ago'] = maize_monthly['mp_price'].shift(12)
maize_monthly['yoy_inflation'] = ((maize_monthly['mp_price'] - maize_monthly['price_12m_ago']) / maize_monthly['price_12m_ago'] * 100)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Prices
ax1.plot(maize_monthly['date'], maize_monthly['mp_price'], linewidth=2.5, color='darkgreen', label='Maize Price')
ax1.set_title('Maize (White) Retail Prices in Kenya', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (KES per KG)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Inflation
ax2.plot(maize_monthly['date'], maize_monthly['yoy_inflation'], linewidth=2.5, color='crimson', label='YoY Inflation')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_title('Year-over-Year Maize Price Inflation', fontsize=14, fontweight='bold')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Inflation Rate (%)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\nMAIZE INFLATION STATISTICS")
print("="*80)
print(f"Average YoY Inflation: {maize_monthly['yoy_inflation'].mean():.2f}%")
print(f"Highest Inflation: {maize_monthly['yoy_inflation'].max():.2f}% ({maize_monthly.loc[maize_monthly['yoy_inflation'].idxmax(), 'date'].strftime('%Y-%m')})")
print(f"Lowest Inflation: {maize_monthly['yoy_inflation'].min():.2f}% ({maize_monthly.loc[maize_monthly['yoy_inflation'].idxmin(), 'date'].strftime('%Y-%m')})")

## 6. Sample Data Exploration

In [ ]:
# Load sample data
sample_data = pd.read_csv('../data/raw/kenya_food_prices_sample.csv')

print("SAMPLE KENYA FOOD PRICES DATASET")
print("="*80)
print(f"Shape: {sample_data.shape}")
print(f"Date Range: {sample_data['date'].min()} to {sample_data['date'].max()}")
print(f"Commodities: {sample_data['commodity'].nunique()}")
print(f"Markets: {sample_data['market'].nunique()}")

print("\nCommodities in sample:")
print(sample_data['commodity'].unique())

print("\nMarkets in sample:")
print(sample_data['market'].unique())

sample_data.head(10)

## 7. Summary and Next Steps

### Key Findings:
1. **WFP Kenya Dataset**: 8,884 records spanning 2006-2021
2. **Top Commodities**: Maize (white), Beans (dry), Rice, Vegetable oil
3. **Geographic Coverage**: 45 markets across 6 regions
4. **Major Markets**: Nairobi, Eldoret, Kisumu, Kitui, Mombasa

### Data Quality:
- Check for missing values in key columns
- Verify price consistency across markets
- Identify outliers and anomalies

### Next Steps:
1. **Data Cleaning**: Handle missing values, outliers
2. **Feature Engineering**: Create inflation indicators, seasonal features
3. **Time Series Analysis**: ARIMA, Prophet, SARIMA models
4. **Spatial Analysis**: Map price variations across regions
5. **Dashboard Development**: Interactive Streamlit app
6. **Forecasting**: Build price prediction models